In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

C:\Users\DELL\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
import akshare as ak

# 抓取沪深300历史行情
df = ak.stock_zh_index_daily(symbol="sh000300")
df.columns = ['date', 'open', 'close', 'high', 'low', 'volume']
df['date'] = pd.to_datetime(df['date'])

print(f"数据抓取成功！拿到 {len(df)} 行行情。")

数据抓取成功！拿到 5881 行行情。


In [ ]:
df.to_csv('quant_data_v1.csv', index=False)
print("已经存为 quant_data_v1.csv")

In [5]:
# --- 1. 数据预处理与特征工程 ---
# 假设 df 已经由前面的 Cell 抓取好了 (含有 date, open, close, high, low, volume)
# 我们手动计算 MACD, RSI 和 MA5
def prepare_data(df):
    df = df.copy()
    # 计算 MA5
    df['MA5'] = df['close'].rolling(window=5).mean()
    
    # 计算 MACD (12, 26, 9)
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd_line'] = ema12 - ema26
    df['signal_line'] = df['macd_line'].ewm(span=9, adjust=False).mean()
    
    # 计算 RSI (14日)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df['rsi_14'] = 100 - (100 / (1 + rs))
    
    # 计算 Target: 下一日收益率 (非常关键，这是模型要预测的靶子)
    df['target'] = df['close'].shift(-1) / df['close'] - 1
    
    df.dropna(inplace=True)
    return df

df_proc = prepare_data(df)

# --- 2. 归一化与数据集封装 ---
FEATURE_COLS = ['close', 'volume', 'MA5', 'macd_line', 'signal_line', 'rsi_14']
scaler = StandardScaler()
df_proc[FEATURE_COLS] = scaler.fit_transform(df_proc[FEATURE_COLS])

class StockDataset(Dataset):
    def __init__(self, data, features, seq_len=30):
        self.X = data[features].values.astype(np.float32)
        self.y = data['target'].values.astype(np.float32)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx : idx + self.seq_len]), 
                torch.tensor(self.y[idx + self.seq_len]))

dataset = StockDataset(df_proc, FEATURE_COLS)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# --- 3. 定义 QuantLSTM 模型 ---
class QuantLSTM(nn.Module):
    def __init__(self, input_sz, hidden_sz, num_layers):
        super(QuantLSTM, self).__init__()
        self.lstm = nn.LSTM(input_sz, hidden_sz, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_sz, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]) # 只取序列最后一天

model = QuantLSTM(len(FEATURE_COLS), 64, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# --- 4. 训练循环 (修复维度 Bug) ---
print("🔥 开始炼丹...")
model.train()
for epoch in range(50): # 增加到50轮，给新特征充分学习时间
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        # 核心修复：labels.view(-1, 1) 确保维度对齐，避免 1633 这种巨额 Loss
        loss = criterion(outputs, labels.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Loss: {total_loss/len(train_loader):.8f}")

# --- 5. 计算 Rank IC (开奖时刻) ---
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in train_loader:
        preds = model(inputs)
        all_preds.extend(preds.squeeze().numpy())
        all_labels.extend(labels.numpy())

rank_ic = pd.Series(all_preds).rank().corr(pd.Series(all_labels).rank())
print(f"\n🏆 最终战果 - Rank IC: {rank_ic:.4f}")

🔥 开始炼丹...
Epoch [10/50], Loss: 0.00018516
Epoch [20/50], Loss: 0.00018322
Epoch [30/50], Loss: 0.00018100
Epoch [40/50], Loss: 0.00018136
Epoch [50/50], Loss: 0.00017971

🏆 最终战果 - Rank IC: 0.1195
